# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable labels, meaningful colours


## Setup — Run this cell first


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# World Happiness Report 2023
# The uploaded WHR2023.csv contains the raw WHR columns.
# The exercise uses a simplified dataframe with a Region column.

raw = pd.read_csv('/mnt/data/WHR2023.csv')

region_map = {
    "Western Europe": ["Austria","Belgium","Cyprus","Denmark","Finland","France","Germany","Greece","Iceland","Ireland","Italy","Luxembourg","Malta","Netherlands","Norway","Portugal","Spain","Sweden","Switzerland","United Kingdom"],
    "Latin America": ["Argentina","Bolivia","Brazil","Chile","Colombia","Costa Rica","Cuba","Dominican Republic","Ecuador","El Salvador","Guatemala","Haiti","Honduras","Jamaica","Mexico","Nicaragua","Panama","Paraguay","Peru","Uruguay","Venezuela"],
    "East Asia": ["China","Hong Kong S.A.R. of China","Japan","Mongolia","South Korea","Taiwan Province of China"],
    "Sub-Saharan Africa": ["Angola","Benin","Botswana","Burkina Faso","Burundi","Cameroon","Central African Republic","Chad","Comoros","Congo (Brazzaville)","Congo (Kinshasa)","Côte d'Ivoire","Djibouti","Eswatini","Ethiopia","Gabon","Gambia","Ghana","Guinea","Guinea-Bissau","Kenya","Lesotho","Liberia","Madagascar","Malawi","Mali","Mauritania","Mauritius","Mozambique","Namibia","Niger","Nigeria","Rwanda","Senegal","Sierra Leone","Somalia","South Africa","South Sudan","Sudan","Tanzania","Togo","Uganda","Zambia","Zimbabwe"],
    "South Asia": ["Afghanistan","Bangladesh","Bhutan","India","Maldives","Nepal","Pakistan","Sri Lanka"],
    "Central and Eastern Europe": ["Albania","Armenia","Azerbaijan","Belarus","Bosnia and Herzegovina","Bulgaria","Croatia","Czechia","Estonia","Georgia","Hungary","Kazakhstan","Kyrgyzstan","Latvia","Lithuania","Moldova","Montenegro","North Macedonia","Poland","Romania","Russia","Serbia","Slovakia","Slovenia","Tajikistan","Turkmenistan","Ukraine","Uzbekistan"],
    "Middle East and North Africa": ["Algeria","Bahrain","Egypt","Iran","Iraq","Israel","Jordan","Kuwait","Lebanon","Libya","Morocco","Oman","Palestine","Qatar","Saudi Arabia","Syria","Tunisia","Türkiye","Turkey","United Arab Emirates","Yemen"],
    "Southeast Asia": ["Cambodia","Indonesia","Laos","Malaysia","Myanmar","Philippines","Singapore","Thailand","Vietnam"],
    "North America and ANZ": ["Australia","Canada","New Zealand","United States"]
}

country_to_region = {
    country: region
    for region, countries in region_map.items()
    for country in countries
}

df = pd.DataFrame({
    'Country': raw['Country name'],
    'Happiness_Score': raw['Ladder score'],
    'GDP': raw['Logged GDP per capita'],
    'Social_Support': raw['Social support'],
    'Life_Expectancy': raw['Healthy life expectancy'],
    'Freedom': raw['Freedom to make life choices'],
    'Generosity': raw['Generosity'],
    'Corruption': raw['Perceptions of corruption']
})
df['Region'] = df['Country'].map(country_to_region)
df = df.dropna(subset=['Region']).reset_index(drop=True)
df = df[['Country','Region','Happiness_Score','GDP','Social_Support',
         'Life_Expectancy','Freedom','Generosity','Corruption']]

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


Dataset: 133 countries, 9 columns
       Country                        Region  ...  Generosity  Corruption
0      Finland                Western Europe  ...      -0.019       0.182
1      Denmark                Western Europe  ...       0.134       0.196
2      Iceland                Western Europe  ...       0.211       0.668
3       Israel  Middle East and North Africa  ...      -0.023       0.708
4  Netherlands                Western Europe  ...       0.213       0.379

[5 rows x 9 columns]


In [2]:
# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())

print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())

print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Sub-Saharan Africa              32
Central and Eastern Europe      25
Western Europe                  20
Latin America                   19
Middle East and North Africa    12
Southeast Asia                   9
East Asia                        6
South Asia                       6
North America and ANZ            4
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
              Country                        Region  Happiness_Score
132       Afghanistan                    South Asia            1.859
131           Lebanon  Middle East and North Africa            2.392
130      Sierra Leone            Sub-Saharan Africa            3.138
129          Zimbabwe            Sub-Saharan Africa            3.204
128  Congo (Kinshasa)            Sub-Saharan Africa            3.207
127          Botswana            Sub-Saharan Africa            3.435
126            Malawi            Sub-Saharan Africa            3.495
125           Comoros         

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation
- Sorted by score, descending
- Zero baseline on x-axis
- At least one design choice beyond the Plotly default
- An insight title that tells the story


In [3]:
# Task 1: Regional comparison bar chart

region_avg = (
    df.groupby('Region')['Happiness_Score']
      .mean()
      .reset_index()
      .sort_values('Happiness_Score', ascending=False)
)

print(region_avg)

fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    text='Happiness_Score',
    title='Western Europe stands out with the highest average happiness score',
    labels={'Happiness_Score': 'Average Happiness Score', 'Region': 'Region'}
)

fig.update_traces(
    marker_color='#4C78A8',
    texttemplate='%{text:.2f}',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Average score: %{x:.2f}<extra></extra>'
)

fig.update_xaxes(rangemode='tozero', showgrid=True, zeroline=True)
fig.update_layout(
    template='plotly_white',
    height=550,
    margin=dict(l=180, r=90, t=90, b=60),
    title=dict(x=0.5, xanchor='center')
)

fig.show()


                         Region  Happiness_Score
4         North America and ANZ         7.018250
8                Western Europe         6.894350
2                 Latin America         5.966053
1                     East Asia         5.930167
0    Central and Eastern Europe         5.928840
6                Southeast Asia         5.431222
3  Middle East and North Africa         5.159000
7            Sub-Saharan Africa         4.361531
5                    South Asia         4.089000


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**.

**Requirements:**
- Show the top 8 AND bottom 8 countries together
- Use colour to distinguish the two groups
- Add a visual separator or annotation
- Use an insight title that tells the story of the gap


In [4]:
# Task 2: Top 8 vs Bottom 8 contrast

top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'

bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()

print("Global average:", round(global_avg, 2))
print(combined[['Country', 'Happiness_Score', 'Group']])

fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    text='Happiness_Score',
    category_orders={'Country': combined['Country'].tolist()},
    color_discrete_map={'Bottom 8': '#D95F59', 'Top 8': '#2A9D8F'},
    title='The happiest countries score far above the least happy 8',
    labels={'Happiness_Score': 'Happiness Score', 'Country': 'Country', 'Group': 'Group'}
)

fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Score: %{x:.2f}<br>%{fullData.name}<extra></extra>'
)

fig.update_xaxes(rangemode='tozero', showgrid=True, zeroline=True)

# Visual separator between the bottom 8 and top 8.
fig.add_shape(
    type='line',
    x0=0, x1=1, xref='paper',
    y0=7.5, y1=7.5,
    line=dict(color='black', width=2, dash='dash')
)

# Global-average reference line.
fig.add_vline(
    x=global_avg,
    line_dash='dot',
    line_width=2,
    annotation_text=f'Global average: {global_avg:.2f}',
    annotation_position='top'
)

fig.update_layout(
    template='plotly_white',
    height=700,
    margin=dict(l=170, r=110, t=100, b=60),
    title=dict(x=0.5, xanchor='center'),
    legend_title_text='Country group'
)

fig.show()


Global average: 5.55
              Country  Happiness_Score     Group
132       Afghanistan            1.859  Bottom 8
131           Lebanon            2.392  Bottom 8
130      Sierra Leone            3.138  Bottom 8
129          Zimbabwe            3.204  Bottom 8
128  Congo (Kinshasa)            3.207  Bottom 8
127          Botswana            3.435  Bottom 8
126            Malawi            3.495  Bottom 8
125           Comoros            3.545  Bottom 8
7         Switzerland            7.240     Top 8
6              Norway            7.315     Top 8
5              Sweden            7.395     Top 8
4         Netherlands            7.403     Top 8
3              Israel            7.473     Top 8
2             Iceland            7.530     Top 8
1             Denmark            7.586     Top 8
0             Finland            7.804     Top 8


---
## Done? Stretch Goal

**Task 3 (stretch):** Build a grouped bar chart comparing GDP and Freedom across the five requested regions.

Regions included:
`Western Europe`, `Latin America`, `East Asia`, `Sub-Saharan Africa`, `South Asia`


In [5]:
# Stretch goal: grouped bar chart

regions = [
    'Western Europe',
    'Latin America',
    'East Asia',
    'Sub-Saharan Africa',
    'South Asia'
]

stretch_avg = (
    df[df['Region'].isin(regions)]
      .groupby('Region')[['GDP', 'Freedom']]
      .mean()
      .reset_index()
)

# GDP and Freedom have different scales, so normalize GDP for a fair visual comparison.
stretch_avg['GDP_scaled'] = stretch_avg['GDP'] / stretch_avg['GDP'].max()

plot_data = stretch_avg[['Region', 'GDP_scaled', 'Freedom']].melt(
    id_vars='Region',
    var_name='Factor',
    value_name='Average_Value'
)

plot_data['Factor'] = plot_data['Factor'].replace({
    'GDP_scaled': 'GDP (scaled)',
    'Freedom': 'Freedom'
})

fig = px.bar(
    plot_data,
    x='Region',
    y='Average_Value',
    color='Factor',
    barmode='group',
    text='Average_Value',
    color_discrete_map={'GDP (scaled)': '#4C78A8', 'Freedom': '#F58518'},
    title='Regional economic strength and freedom show different patterns',
    labels={'Average_Value': 'Average value (scaled)', 'Region': 'Region'}
)

fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

fig.update_yaxes(rangemode='tozero')
fig.update_layout(
    template='plotly_white',
    height=550,
    margin=dict(l=70, r=60, t=100, b=100),
    title=dict(x=0.5, xanchor='center')
)

fig.show()
